[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/appars/codepilot-colab/blob/main/stage3_tool_agent/Stage3_Tool_Agent.ipynb)

> **Click the badge above to open this notebook in Google Colab.**
> Or go directly: https://colab.research.google.com/github/appars/codepilot-colab/blob/main/stage3_tool_agent/Stage3_Tool_Agent.ipynb

# 🔧 Stage 3 — Tool Agent
**CodePilot AI Studio | Module 4 | Agentic AI in Software Engineering**

---

## What You Will Learn
- What **tool orchestration** means in agentic AI
- Why **specialised prompts** produce far better results than one generic prompt
- How an **intent router** classifies what the user wants automatically
- How a **skill dispatcher** sends the request to the right tool
- The foundation of the **ReAct pattern** (Reasoning + Acting)

## The Big Idea
```
Generic (Stage 2):              Tool Agent (Stage 3):
'Help me with my code'          'debug this code'
        ↓                               ↓
One prompt tries to            Intent Router detects: DEBUG
debug+explain+review                   ↓
→ vague output                 Debug Skill runs focused prompt
                               → structured, excellent output!
```

> **Analogy:** A hospital sends you to a specialist — cardiologist, surgeon, neurologist.
> One expert per job. Not one generic doctor for everything.

⏱ **Expected time: 20 minutes**

## Step 1 — Add Your Groq API Key (One Time Setup)

### Option A — Using Colab Secrets (Recommended!)
Store your key ONCE in Colab Secrets and it works in ALL notebooks automatically:
1. Click the **🔑 key icon** in the left sidebar (or go to Tools → Secrets)
2. Click **'Add new secret'**
3. Name: `GROQ_API_KEY`  (must be exactly this name)
4. Value: paste your key (looks like `gsk_xxxx...`)
5. Toggle **'Notebook access'** to ON
6. Come back and run this cell

### Option B — Paste directly (quick one-time use)
If you do not want to use Secrets, just paste your key in the code cell below.

> Get your free key at **https://console.groq.com** → API Keys → Create API Key

In [ ]:
# ── GROQ API KEY SETUP ───────────────────────────────────────
# This cell tries Colab Secrets first (recommended).
# If not found, falls back to manual paste below.

import os

# ── METHOD 1: Colab Secrets (store once, works in all notebooks)
# If you added GROQ_API_KEY in the Secrets panel, this will find it.
try:
    from google.colab import userdata
    GROQ_API_KEY = userdata.get('GROQ_API_KEY')
    if GROQ_API_KEY:
        print("Groq API key loaded from Colab Secrets!")
        print(f"Key starts with: {GROQ_API_KEY[:8]}...")
    else:
        raise ValueError("Key not found in Secrets")
except Exception as e:
    # ── METHOD 2: Manual paste (fallback)
    # If Secrets is not set up, paste your key here:
    GROQ_API_KEY = "paste-your-groq-key-here"   # ← replace if not using Secrets
    if GROQ_API_KEY == "paste-your-groq-key-here":
        print("ERROR: Key not found in Secrets and not pasted manually.")
        print("")
        print("Option A: Add to Colab Secrets:")
        print("  1. Click the key icon in the left sidebar")
        print("  2. Add secret name: GROQ_API_KEY")
        print("  3. Paste your key as the value")
        print("  4. Enable Notebook access and re-run this cell")
        print("")
        print("Option B: Paste your key directly above (replace 'paste-your-groq-key-here')")
        print("Get your free key from: https://console.groq.com")
    else:
        print(f"Groq API key set manually. Starts with: {GROQ_API_KEY[:8]}...")

# Set as environment variable so LangChain reads it automatically
os.environ["GROQ_API_KEY"] = GROQ_API_KEY

# Final check
if len(GROQ_API_KEY) > 20:
    print("Ready to proceed!")


## Step 2 — Setup

In [ ]:
print("Installing packages...")
!pip install -q langchain-groq langchain langchain-community langchain-core langchain-classic langchain-text-splitters
print("Setup complete!")

## Step 3 — Understand Intent Detection

Before building the full agent, let us understand how the **intent router** works.

The router reads what the user typed and decides which skill to use.

### How keywords work
```python
request = 'debug this code — it crashes'
         ↓
contains 'debug'? YES → return 'debug'
contains 'explain'? → check next
contains 'review'? → check next
```

### The three skills
| Skill | Keywords | What it does |
|-------|----------|--------------|
| debug | debug, fix, error, bug, crash | Finds bug, explains root cause, fixes it |
| explain | explain, what does, understand | Plain English explanation for beginners |
| review | review, improve, better, quality | Code quality review with suggestions |

In [ ]:
# ── CONCEPT DEMO: The Intent Router (no LLM needed) ──────────
# This function reads a request and returns the detected skill.
# Test it with different requests to see how it classifies them.

def detect_intent(user_request: str) -> str:
    """
    Detect what skill the user needs based on keywords.
    Returns: 'debug', 'explain', 'review', or 'unknown'
    """
    # Convert to lowercase so 'Debug' and 'debug' both match
    request_lower = user_request.lower()

    # Keywords for each skill
    debug_keywords   = ["debug", "fix", "error", "bug", "broken",
                        "not working", "crash", "exception", "wrong"]
    explain_keywords = ["explain", "what does", "understand", "how does",
                        "describe", "walk me through", "what is"]
    review_keywords  = ["review", "improve", "better", "quality",
                        "best practice", "feedback", "clean", "refactor"]

    # Check each list — return on first match
    # any() returns True if ANY keyword in the list is found
    if any(kw in request_lower for kw in debug_keywords):
        return "debug"
    elif any(kw in request_lower for kw in explain_keywords):
        return "explain"
    elif any(kw in request_lower for kw in review_keywords):
        return "review"
    else:
        return "unknown"   # fallback when no keyword matches

# Test the router with different requests
test_requests = [
    "debug this code — it crashes",
    "can you explain what this function does?",
    "please review my code and give feedback",
    "help me with my Python code",    # no clear keyword → unknown
    "there is a bug in line 5!",
    "I don't understand this loop",
]

print("Intent Router Test (no LLM needed):")
print("-" * 55)
for req in test_requests:
    intent = detect_intent(req)
    print(f"  Request : '{req}'")
    print(f"  Intent  : {intent}")
    print()
print("The same function will automatically route real user requests!")

## Step 4 — The Full Tool Agent
Now let us wire the intent router to three specialised skill prompts and a real LLM.

In [ ]:
# ============================================================
# CodePilot AI Studio — Stage 3: Tool Agent
# ============================================================
# Concept  : Tool orchestration — intent detection + skill dispatch
# New here : detect_intent(), 3 focused prompts, run_skill()
# ============================================================

from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate

# ── STEP 1: Connect to LLM ────────────────────────────────────
# temperature=0.3 → more precise for code tasks (less random)
llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0.3)

# ── STEP 2: Define three specialised skill prompts ─────────────
# Each prompt gives Llama 3 a DIFFERENT role and DIFFERENT structure.
# Same model, completely different output quality per skill!

# Skill 1: Debug — finds the bug, explains root cause, provides fix
DEBUG_PROMPT = PromptTemplate(
    input_variables=["code"],
    template="""You are an expert Python debugger.
Analyse this code very carefully.

Your response MUST follow this exact structure:
1. BUG FOUND: [describe the bug in one clear sentence]
2. WHY IT HAPPENS: [explain the root cause simply]
3. FIXED CODE: [show the complete corrected code]
4. EXPLANATION: [explain what changed and why]

Code to debug:
```python
{code}
```"""
)

# Skill 2: Explain — plain English explanation for beginners
EXPLAIN_PROMPT = PromptTemplate(
    input_variables=["code"],
    template="""You are a patient Python tutor explaining to a complete beginner.

Your response MUST follow this exact structure:
1. WHAT IT DOES: [one sentence summary]
2. HOW IT WORKS: [step by step in plain English — no jargon]
3. KEY CONCEPTS: [Python concepts used in this code]
4. BEGINNER TIP: [one practical advice for a first-year student]

Code to explain:
```python
{code}
```"""
)

# Skill 3: Review — code quality feedback from a senior developer
REVIEW_PROMPT = PromptTemplate(
    input_variables=["code"],
    template="""You are a senior Python developer doing a thorough code review.

Your response MUST follow this exact structure:
1. OVERALL QUALITY: [X/10 with one sentence reason]
2. ISSUES FOUND: [list every issue clearly]
3. IMPROVEMENTS: [specific code suggestions]
4. POSITIVE ASPECTS: [what is done well]

Code to review:
```python
{code}
```"""
)

# ── STEP 3: The Skill Dispatcher ──────────────────────────────
# Takes the detected intent + code and runs the right skill
def run_skill(intent: str, code: str) -> str:
    """
    Route to the correct skill based on detected intent.
    Each skill uses a focused prompt for the best results.
    """
    if intent == "debug":
        print("  Routing to: Debug Skill")
        # .format() fills in the {code} placeholder in the prompt
        prompt = DEBUG_PROMPT.format(code=code)
    elif intent == "explain":
        print("  Routing to: Explain Skill")
        prompt = EXPLAIN_PROMPT.format(code=code)
    elif intent == "review":
        print("  Routing to: Review Skill")
        prompt = REVIEW_PROMPT.format(code=code)
    else:
        print("  Intent unknown — using generic prompt")
        prompt = f"Help with this Python code:\n```python\n{code}\n```"

    response = llm.invoke(prompt)
    return response.content

# ── STEP 4: Test all three skills on the same buggy code ───────
SAMPLE_CODE = """
def calculate_average(numbers):
    total = 0
    for num in numbers:
        total = total + num
    average = total / len(numbers)   # BUG: crashes when list is empty!
    return average

print(calculate_average([85, 92, 78]))   # works fine
print(calculate_average([]))             # ZeroDivisionError!
"""

print("=" * 60)
print("CodePilot AI Studio — Stage 3: Tool Agent")
print("=" * 60)
print()
print("Testing all 3 skills on the SAME code:")
print(SAMPLE_CODE)

# Test 1: Debug
req1 = "This code crashes when I pass an empty list. Can you debug it?"
print(f"Request: '{req1}'")
intent1 = detect_intent(req1)
print(f"  Detected intent: {intent1}")
print(f"  CodePilot (Debug Skill):")
print(run_skill(intent1, SAMPLE_CODE))
print()

# Test 2: Explain
req2 = "Can you explain what this code does?"
print("-" * 60)
print(f"Request: '{req2}'")
intent2 = detect_intent(req2)
print(f"  Detected intent: {intent2}")
print(f"  CodePilot (Explain Skill):")
print(run_skill(intent2, SAMPLE_CODE))
print()

# Test 3: Review
req3 = "Please review my code and give feedback."
print("-" * 60)
print(f"Request: '{req3}'")
intent3 = detect_intent(req3)
print(f"  Detected intent: {intent3}")
print(f"  CodePilot (Review Skill):")
print(run_skill(intent3, SAMPLE_CODE))
print()
print("=" * 60)
print("KEY INSIGHT: Same code, 3 different requests → 3 very different outputs.")
print("The intent router picked the right specialist automatically!")

## Step 5 — Verify

In [ ]:
# Verify all 3 skills ran
try:
    tests = [
        detect_intent("debug this crash") == "debug",
        detect_intent("explain this code") == "explain",
        detect_intent("review my code") == "review",
    ]
    if all(tests):
        print("VERIFICATION PASSED")
        print("  detect_intent('debug this crash')  → debug")
        print("  detect_intent('explain this code') → explain")
        print("  detect_intent('review my code')    → review")
        print("Ready for Stage 4 (homework)!")
    else:
        print("Something failed. Check detect_intent() function.")
except:
    print("Run Step 4 first.")

## Step 6 — Experiments

In [ ]:
# ── EXPERIMENT 1: Test with your own code ─────────────────────
# Paste YOUR Python code below and change my_request.

my_code = """
def find_student(students, name):
    for student in students:
        if student['name'] == name:
            return student
    # what if student not found? returns None silently!

class_list = [{'name': 'Alice', 'grade': 90}]
result = find_student(class_list, 'Bob')  # Bob not in list!
print(result['grade'])   # crashes here!
"""

# Change this to test different skills:
# 'debug this code', 'explain what this does', 'review my code'
my_request = "debug this code"

my_intent = detect_intent(my_request)
print(f"Request : '{my_request}'")
print(f"Intent  : {my_intent}")
print(f"\nCodePilot says:")
print(run_skill(my_intent, my_code))

In [ ]:
# ── EXPERIMENT 2: Add an 'optimize' skill ─────────────────────
# Step 1: Add optimize keywords to detect_intent()
# Step 2: Create an OPTIMIZE_PROMPT
# Step 3: Add 'optimize' case to run_skill()

# Here is a starter — complete it!

OPTIMIZE_PROMPT = PromptTemplate(
    input_variables=["code"],
    template="""You are a Python performance expert.
Review this code for performance issues and suggest optimizations.

1. PERFORMANCE ISSUES: [list slow or inefficient parts]
2. OPTIMIZED CODE: [show the improved version]
3. WHY IT IS FASTER: [explain the improvement]

Code:
```python
{code}
```"""
)

# Test the new skill directly
slow_code = """
numbers = [1, 2, 3, 4, 5]
total = 0
for n in numbers:       # slow: manual loop
    total = total + n
print(total)
"""

result = llm.invoke(OPTIMIZE_PROMPT.format(code=slow_code))
print("=== Optimize Skill Result ===")
print(result.content)
print()
print("Now try adding 'optimize' to detect_intent() so it auto-routes!")

In [ ]:
# ── EXPERIMENT 3: Use LLM to detect intent ────────────────────
# Right now intent detection uses simple keywords.
# What if we used the LLM itself to classify the request?
# This is more flexible but uses an extra API call.

def detect_intent_with_llm(user_request: str) -> str:
    """Use Llama 3 to classify the intent instead of keywords."""
    classifier_prompt = f"""Classify this request into exactly one category.
Categories: debug, explain, review, unknown
Reply with ONLY the category word, nothing else.

Request: '{user_request}'
Category:"""

    response = llm.invoke(classifier_prompt)
    return response.content.strip().lower()

# Test both methods
test_cases = [
    "my code has a strange issue",
    "I am confused about this function",
    "can you make this better?",
]

print("Comparing keyword detection vs LLM detection:")
print("-" * 55)
for req in test_cases:
    kw_result  = detect_intent(req)
    llm_result = detect_intent_with_llm(req)
    print(f"Request  : '{req}'")
    print(f"Keywords : {kw_result}")
    print(f"LLM      : {llm_result}")
    print()
print("Which method is more accurate? Which is faster?")

## Summary — Stage 3 Complete!

| Concept | What it means |
|---------|---------------|
| **Tool orchestration** | Routing tasks to specialised skills |
| **Intent detection** | Classifying what the user wants |
| **`detect_intent()`** | Reads keywords → returns skill name |
| **`PromptTemplate`** | Focused prompt with fill-in {code} variable |
| **`run_skill()`** | Dispatches to the right prompt + LLM call |
| **ReAct pattern** | Reasoning (detect intent) + Acting (run skill) |

---
## Stages 4–6 (Homework)

| Stage | Concept | What it adds |
|-------|---------|-------------|
| **Stage 4** | RAG | Long-term memory — retrieves relevant knowledge before answering |
| **Stage 5** | Reflection | Agent critiques and improves its own output automatically |
| **Stage 6** | LangGraph | Stateful graph workflow — agent can loop back and retry |

---
**Stages 1–3 complete! You have built a real AI coding assistant.**
Continue with Stage 4 at home: `stage4_rag_explain/Stage4_RAG_Explain.ipynb`